# Assignment 1 - Classification

Group Members
1. GEROME VINCENT CARREON
2. NIEL FRANCIS ARLIGUE
3. RAPHAEL NICOLAI GIGANTE
4. TRENT JOAQIN SANTOS

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Adult_Census_Income.csv")
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [3]:
(df == "?").sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64

## Question 1A: Identify a Numerical Dataset Suitable for Classification

The dataset used is the Adult Census Income dataset. It contains demographic and employment-related information such as age, education, occupation, capital gain, capital loss, and working hours per week.

This dataset is suitable for classification because the goal is to predict whether a person's income is `<=50K` or `>50K`.

The dataset includes numerical features such as `age`, `fnlwgt`, `education.num`, `capital.gain`, `capital.loss`, and `hours.per.week`.

The dataset is noisy because some missing or irregular values are represented by `?`. These appear in `workclass`, `occupation`, and `native.country`.

In [5]:
df = df.replace("?", np.nan)

df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64

In [6]:
df["income"] = df["income"].str.strip()

df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

df["income"].value_counts()

income
0    24720
1     7841
Name: count, dtype: int64

## Question 1B: Data Cleaning

The dataset was cleaned by replacing `?` with missing values. 

Missing values are present in:
- workclass
- occupation
- native.country

These will be handled using imputation:
- numerical columns → median
- categorical columns → most frequent value

The target column `income` was converted into:
- 0 for <=50K
- 1 for >50K

In [7]:
X = df.drop("income", axis=1)
y = df["income"]

X.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
0,90,NaN,77053,HS-grad,9,Widowed,NaN,Not-in-family,White,Female,0,4356,40,United-States
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States
2,66,NaN,186061,Some-college,10,Widowed,NaN,Unmarried,Black,Female,0,4356,40,United-States
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

numeric_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(include=["str"]).columns

numeric_features, categorical_features

(Index(['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss',
        'hours.per.week'],
       dtype='str'),
 Index(['workclass', 'education', 'marital.status', 'occupation',
        'relationship', 'race', 'sex', 'native.country'],
       dtype='str'))

In [9]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## Question 1C: Machine Learning Algorithm

Random Forest Classifier was used because it works well for classification tasks and can handle complex relationships between features after preprocessing.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_estimators=300,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:")
print(conf_matrix)
print(classification_report(y_test, y_pred))

Accuracy: 0.8507600184246891
Confusion Matrix:
[[4587  358]
 [ 614  954]]
              precision    recall  f1-score   support

           0       0.88      0.93      0.90      4945
           1       0.73      0.61      0.66      1568

    accuracy                           0.85      6513
   macro avg       0.80      0.77      0.78      6513
weighted avg       0.84      0.85      0.85      6513



try gradient boosting

Unlike random forest, which builds independent trees, gradient boosting builds trees sequentially to continuously correct and minimize the errors of previous trees.

In [11]:
# try Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        random_state=42,
        n_estimators=100
    ))
])

gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)

accuracy_gb = accuracy_score(y_test, y_pred_gb)
conf_matrix_gb = confusion_matrix(y_test, y_pred_gb)

print("Gradient Boosting Accuracy:", accuracy_gb)
print("Gradient Boosting Confusion Matrix:")
print(conf_matrix_gb)
print(classification_report(y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.8607400583448488
Gradient Boosting Confusion Matrix:
[[4675  270]
 [ 637  931]]
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      4945
           1       0.78      0.59      0.67      1568

    accuracy                           0.86      6513
   macro avg       0.83      0.77      0.79      6513
weighted avg       0.85      0.86      0.85      6513



Random forest classifier: 85.07%
Gradient boost: 86.07 %

## Parameter Tuning

Parameter tuning was tested to check if the accuracy could improve further.

In [12]:
tuned_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_estimators=500,
        max_depth=20,
        min_samples_split=4,
        class_weight="balanced"
    ))
])

tuned_model.fit(X_train, y_train)

y_pred_tuned = tuned_model.predict(X_test)

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
conf_matrix_tuned = confusion_matrix(y_test, y_pred_tuned)

print("Tuned Accuracy:", accuracy_tuned)
print("Tuned Confusion Matrix:")
print(conf_matrix_tuned)
print(classification_report(y_test, y_pred_tuned))

Tuned Accuracy: 0.826654383540611
Tuned Confusion Matrix:
[[4123  822]
 [ 307 1261]]
              precision    recall  f1-score   support

           0       0.93      0.83      0.88      4945
           1       0.61      0.80      0.69      1568

    accuracy                           0.83      6513
   macro avg       0.77      0.82      0.79      6513
weighted avg       0.85      0.83      0.83      6513



In [13]:
# Parameter tuning for gradient boost
tuned_gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        min_samples_split=4,
        random_state=42
    ))
])

tuned_gb_model.fit(X_train, y_train)

y_pred_tuned_gb = tuned_gb_model.predict(X_test)

accuracy_tuned_gb = accuracy_score(y_test, y_pred_tuned_gb)
conf_matrix_tuned_gb = confusion_matrix(y_test, y_pred_tuned_gb)

print("Tuned Gradient Boosting Accuracy:", accuracy_tuned_gb)
print("Tuned Gradient Boosting Confusion Matrix:")
print(conf_matrix_tuned_gb)
print(classification_report(y_test, y_pred_tuned_gb))

Tuned Gradient Boosting Accuracy: 0.8708736373407032
Tuned Gradient Boosting Confusion Matrix:
[[4647  298]
 [ 543 1025]]
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4945
           1       0.77      0.65      0.71      1568

    accuracy                           0.87      6513
   macro avg       0.84      0.80      0.81      6513
weighted avg       0.87      0.87      0.87      6513



- Tuned Random Forest: 82.67% which is lower than the default Random Forest
- Tuned Gradient Boost: 87.09% which is higher than the default Gradient Boost


The confusion matrix shows:
- Class 1 (>50K) is predicted well with high precision and recall
- Class 0 (<=50K) has lower precision and recall

This is due to class imbalance in the dataset, where most samples belong to the <=50K class.

Because of this imbalance, the model is biased toward predicting the majority class.

- - -

The model did not reach the expected 90% accuracy. This is because the Adult Census dataset is more complex and contains noisy categorical features, making it harder to classify perfectly.

Despite tuning, the model performance remained below 90%, which reflects the realistic difficulty of the dataset.

## Question 2: Discussion on random_state Parameter

The `random_state` parameter controls the randomness of the train-test split. When it is fixed, the same training and testing data are used each time. When it is removed, the split changes each run, which can produce different accuracy scores.

In [14]:
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y
)

model_no_random_1 = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model_no_random_1.fit(X_train_1, y_train_1)

y_pred_1 = model_no_random_1.predict(X_test_1)

accuracy_1 = accuracy_score(y_test_1, y_pred_1)

print("Accuracy without random_state, Run 1:", accuracy_1)

Accuracy without random_state, Run 1: 0.8562874251497006


In [15]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y
)

model_no_random_2 = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model_no_random_2.fit(X_train_2, y_train_2)

y_pred_2 = model_no_random_2.predict(X_test_2)

accuracy_2 = accuracy_score(y_test_2, y_pred_2)

print("Accuracy without random_state, Run 2:", accuracy_2)

Accuracy without random_state, Run 2: 0.8587440503608168


The two runs produced different accuracy scores:

- Run 1: 85.62%
- Run 2: 85.87%

The scores are different because `random_state` was removed, so the train-test split changed each time. This shows that `random_state` is important for making machine learning results reproducible.